# Phase 1: Data Infrastructure & Empirical Diagnostics  
## Notebook 01_09 — Tick-to-Bar Sampling Bias with L1 Bid/Ask Data

### Research question

How does converting L1 tick-level quote data into fixed bars change measured returns, volatility, spread, and signal quality?

This notebook is designed for the realistic data constraint from the internship project:

> The available Chinese futures data contains L1 top-of-book quotes — `bid1` and `ask1` — rather than full limit order book depth.

That means this notebook **does not** claim to model the full order book. It focuses on what can be studied honestly with best bid and best ask data:

1. mid-price construction;
2. quoted spread measurement;
3. quote update intensity;
4. time-bar sampling;
5. quote-count sampling;
6. range / price-move bars;
7. previous-tick versus next-tick interpolation;
8. bid/ask bounce and why mid-prices are safer than raw bid or ask;
9. volatility signature plots;
10. session-break handling for Chinese futures-style day/night sessions;
11. limits of L1 data compared with full L2/L3 order-book data.

The goal is to build a robust tick-to-bar layer for later futures alpha, volatility, and backtesting notebooks without overstating what the data can support.

## 1. Why this notebook matters

Many quant models are trained on bars:

- 1-minute bars;
- 5-minute bars;
- 30-minute bars;
- daily bars;
- event bars;
- range bars.

But the market generates events irregularly.

If we compress tick data into bars carelessly, we can create distorted estimates of:

- returns;
- volatility;
- autocorrelation;
- spreads;
- trading intensity;
- liquidity;
- signal decay.

This is especially important for futures because activity is not uniform across the day. Chinese commodity futures may have day sessions, night sessions, lunch breaks, product-specific hours, and exchange-specific microstructure rules.

The central warning is:

> Bars are not raw data. Bars are a sampling choice.

## 2. What L1 data can and cannot support

### Available with `bid1` and `ask1`

With best bid and best ask prices, we can compute:

$$
\text{mid}_t = \frac{\text{bid1}_t + \text{ask1}_t}{2}
$$
$$
\text{spread}_t = \text{ask1}_t - \text{bid1}_t
$$
$$
\text{relative spread}_t = \frac{\text{ask1}_t - \text{bid1}_t}{\text{mid}_t}
$$
We can also study:

- quote update frequency;
- spread regimes;
- mid-price returns;
- volatility signature plots;
- quote staleness;
- locked or crossed quote anomalies;
- time-bar versus event-bar sampling;
- session-gap handling.

### Not available without deeper data

Without full L2/L3 order book depth, we should **not** claim to know:

- queue position;
- depth at levels 2, 3, 4, 5, etc.;
- full order imbalance;
- market order flow;
- hidden liquidity;
- order cancellations throughout the book;
- true microprice if bid/ask sizes are missing;
- fill probability from queue dynamics.

If bid/ask sizes are available, a simple top-of-book imbalance can be computed. If only prices are available, even that should be excluded or treated as optional.

## 3. Key definitions

For each quote update $i$, we observe:

$$
(t_i, b_i, a_i)
$$
where:

- $t_i$ is the quote timestamp;
- $b_i$ is best bid price;
- $a_i$ is best ask price.

The mid-price is:

$$
m_i = \frac{a_i + b_i}{2}
$$
The quoted spread is:

$$
s_i = a_i - b_i
$$
The log mid-price return between two sampled observations is:

$$
r_i = \log m_i - \log m_{i-1}
$$
The key question is:

> Which observations should define the sampled series?

## 4. Imports and configuration

The notebook uses synthetic L1 quote data so it can run without private internship data.

Later, a real data loader can replace the synthetic generator as long as it returns at least:

```text
timestamp, bid1, ask1
```

Optional fields such as `bid1_size`, `ask1_size`, `last_price`, and `volume` can be added later.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
@dataclass(frozen=True)
class L1SimulationConfig:
    """
    Synthetic L1 quote simulation configuration.

    The session windows are stylised and should be replaced with product-specific
    exchange calendars for real Chinese futures data.
    """
    start_date: str = "2024-01-02"
    num_business_days: int = 12
    symbol: str = "SYN_FUT"
    tick_size: float = 1.0
    base_price: float = 3_000.0
    annual_vol: float = 0.30
    min_spread_ticks: int = 1
    max_spread_ticks: int = 5
    seed: int = 42


config = L1SimulationConfig()
config

## 5. Synthetic Chinese-futures-style sessions

For demonstration, we use stylised sessions:

| Session | Time |
|---|---|
| Night | 21:00–23:00 |
| Morning 1 | 09:00–10:15 |
| Morning 2 | 10:30–11:30 |
| Afternoon | 13:30–15:00 |

These are illustrative, not a universal exchange calendar.

Real Chinese futures products can have different night-session hours and exchange-specific rules. For production research, session calendars should be product-specific metadata rather than hard-coded assumptions.

In [ ]:
SESSION_WINDOWS = [
    ("night", "21:00:00", "23:00:00"),
    ("morning_1", "09:00:00", "10:15:00"),
    ("morning_2", "10:30:00", "11:30:00"),
    ("afternoon", "13:30:00", "15:00:00"),
]


def business_dates(start_date: str, num_business_days: int) -> pd.DatetimeIndex:
    """
    Generate business dates.
    """
    return pd.bdate_range(start_date, periods=num_business_days)


def make_session_start_end(date: pd.Timestamp, start_time: str, end_time: str) -> tuple[pd.Timestamp, pd.Timestamp]:
    """
    Create timezone-naive session start/end timestamps for a date.
    """
    start = pd.Timestamp(f"{date.date()} {start_time}")
    end = pd.Timestamp(f"{date.date()} {end_time}")
    return start, end

## 6. Simulating irregular L1 quote updates

Real tick data is irregular.

During active periods, quotes update frequently. During quiet periods, quotes update slowly.

We simulate quote timestamps using random inter-arrival times, then simulate an efficient mid-price and a noisy quoted spread.

The observed bid and ask are rounded to the tick grid.

In [ ]:
def round_to_tick(price: np.ndarray | float, tick_size: float) -> np.ndarray | float:
    """
    Round price to nearest tick.
    """
    return np.round(np.asarray(price) / tick_size) * tick_size


def simulate_session_quotes(
    session_date: pd.Timestamp,
    session_name: str,
    start_time: str,
    end_time: str,
    start_price: float,
    config: L1SimulationConfig,
    local_rng: np.random.Generator
) -> tuple[pd.DataFrame, float]:
    """
    Simulate one session of L1 bid/ask quote updates.
    """
    start_ts, end_ts = make_session_start_end(session_date, start_time, end_time)

    session_seconds = int((end_ts - start_ts).total_seconds())

    # Quote intensity varies by session.
    if session_name in ["morning_1", "night"]:
        mean_interarrival = 2.0
    elif session_name == "afternoon":
        mean_interarrival = 3.0
    else:
        mean_interarrival = 4.0

    interarrivals = local_rng.exponential(scale=mean_interarrival, size=int(session_seconds / mean_interarrival * 1.5))
    event_seconds = np.cumsum(interarrivals)
    event_seconds = event_seconds[event_seconds < session_seconds]

    if len(event_seconds) < 10:
        return pd.DataFrame(), start_price

    timestamps = start_ts + pd.to_timedelta(event_seconds, unit="s")
    n = len(timestamps)

    # Simulate efficient log-price at event times.
    dt_years = np.diff(np.insert(event_seconds, 0, 0.0)) / (252 * 4.5 * 3600)
    shocks = config.annual_vol * np.sqrt(dt_years) * local_rng.standard_normal(n)

    # Add small intraday drift pulses.
    drift = 0.02 * dt_years
    efficient_log = np.log(start_price) + np.cumsum(drift + shocks)
    efficient_price = np.exp(efficient_log)

    # Occasional bursts around synthetic microstructure shocks.
    burst_mask = local_rng.uniform(size=n) < 0.01
    efficient_price[burst_mask] *= np.exp(local_rng.normal(0.0, 0.004, burst_mask.sum()))

    mid = round_to_tick(efficient_price, config.tick_size)

    spread_ticks = local_rng.choice(
        np.arange(config.min_spread_ticks, config.max_spread_ticks + 1),
        size=n,
        p=np.array([0.55, 0.25, 0.12, 0.05, 0.03])[:config.max_spread_ticks] / np.array([0.55, 0.25, 0.12, 0.05, 0.03])[:config.max_spread_ticks].sum()
    )

    spread = spread_ticks * config.tick_size

    bid1 = mid - spread / 2
    ask1 = mid + spread / 2

    # Round bid/ask to tick grid and enforce ask >= bid + tick.
    bid1 = round_to_tick(bid1, config.tick_size)
    ask1 = round_to_tick(ask1, config.tick_size)
    ask1 = np.maximum(ask1, bid1 + config.tick_size)

    # Optional top-of-book sizes, included to show what would be possible if sizes exist.
    bid1_size = local_rng.lognormal(mean=3.0, sigma=0.5, size=n).round()
    ask1_size = local_rng.lognormal(mean=3.0, sigma=0.5, size=n).round()

    out = pd.DataFrame({
        "timestamp": timestamps,
        "symbol": config.symbol,
        "session": session_name,
        "bid1": bid1.astype(float),
        "ask1": ask1.astype(float),
        "bid1_size": bid1_size.astype(float),
        "ask1_size": ask1_size.astype(float),
        "efficient_price": efficient_price,
    })

    end_price = float(efficient_price[-1])

    return out, end_price

In [ ]:
def simulate_l1_quote_panel(config: L1SimulationConfig) -> pd.DataFrame:
    """
    Simulate multiple days of L1 quote data.
    """
    local_rng = np.random.default_rng(config.seed)
    dates = business_dates(config.start_date, config.num_business_days)

    frames = []
    current_price = config.base_price

    for date in dates:
        # Overnight jump into the new business date.
        current_price *= np.exp(local_rng.normal(0.0, 0.004))

        for session_name, start_time, end_time in SESSION_WINDOWS:
            session_df, current_price = simulate_session_quotes(
                session_date=date,
                session_name=session_name,
                start_time=start_time,
                end_time=end_time,
                start_price=current_price,
                config=config,
                local_rng=local_rng
            )

            if not session_df.empty:
                frames.append(session_df)

            # Session gap jump.
            current_price *= np.exp(local_rng.normal(0.0, 0.002))

    quotes = pd.concat(frames, ignore_index=True)
    quotes = quotes.sort_values("timestamp").reset_index(drop=True)

    return quotes


quotes = simulate_l1_quote_panel(config)

quotes.head()

In [ ]:
quotes.shape

## 7. L1 quote validation and derived fields

Before sampling, validate top-of-book quotes.

Basic L1 checks:

1. `bid1` and `ask1` are finite;
2. `bid1 > 0` and `ask1 > 0`;
3. `ask1 >= bid1`;
4. spread is not negative;
5. timestamps are sorted;
6. duplicate timestamps are handled;
7. session labels are consistent.

Derived fields:

$$
\text{mid} = \frac{\text{bid1}+\text{ask1}}{2}
$$
$$
\text{spread} = \text{ask1}-\text{bid1}
$$
$$
\text{relative spread} = \frac{\text{spread}}{\text{mid}}
$$

In [ ]:
def validate_l1_quotes(quotes: pd.DataFrame) -> pd.DataFrame:
    """
    Validate L1 bid/ask quote data and return a diagnostic table.
    """
    checks = []

    checks.append({
        "check": "timestamp_sorted",
        "passed": quotes["timestamp"].is_monotonic_increasing,
        "detail": "timestamps are monotonic increasing"
    })

    checks.append({
        "check": "no_duplicate_timestamp_symbol",
        "passed": not quotes.duplicated(["timestamp", "symbol"]).any(),
        "detail": int(quotes.duplicated(["timestamp", "symbol"]).sum())
    })

    for col in ["bid1", "ask1"]:
        finite = np.isfinite(quotes[col]).all()
        positive = (quotes[col] > 0).all()

        checks.append({
            "check": f"{col}_finite",
            "passed": bool(finite),
            "detail": int((~np.isfinite(quotes[col])).sum())
        })

        checks.append({
            "check": f"{col}_positive",
            "passed": bool(positive),
            "detail": int((quotes[col] <= 0).sum())
        })

    crossed = quotes["ask1"] < quotes["bid1"]
    locked = quotes["ask1"] == quotes["bid1"]

    checks.append({
        "check": "no_crossed_quotes",
        "passed": not crossed.any(),
        "detail": int(crossed.sum())
    })

    checks.append({
        "check": "locked_quotes_count",
        "passed": True,
        "detail": int(locked.sum())
    })

    return pd.DataFrame(checks)


validation_report = validate_l1_quotes(quotes)
validation_report

In [ ]:
def add_l1_features(quotes: pd.DataFrame) -> pd.DataFrame:
    """
    Add basic L1 quote-derived fields.
    """
    out = quotes.sort_values(["symbol", "timestamp"]).copy()

    out["mid"] = (out["bid1"] + out["ask1"]) / 2
    out["spread"] = out["ask1"] - out["bid1"]
    out["relative_spread"] = out["spread"] / out["mid"]

    out["log_mid"] = np.log(out["mid"])
    out["mid_return_tick"] = out.groupby("symbol")["log_mid"].diff()

    out["quote_interval_seconds"] = (
        out.groupby("symbol")["timestamp"].diff().dt.total_seconds()
    )

    # Optional top-of-book imbalance only if sizes exist.
    if {"bid1_size", "ask1_size"}.issubset(out.columns):
        denominator = out["bid1_size"] + out["ask1_size"]
        out["top_of_book_imbalance"] = np.where(
            denominator > 0,
            (out["bid1_size"] - out["ask1_size"]) / denominator,
            np.nan
        )

        out["microprice_l1_optional"] = np.where(
            denominator > 0,
            (out["ask1"] * out["bid1_size"] + out["bid1"] * out["ask1_size"]) / denominator,
            np.nan
        )

    return out


quotes_l1 = add_l1_features(quotes)

quotes_l1.head()

## 8. Visualising L1 quote data

The mid-price removes much of the bid/ask bounce that would appear if one alternated between bid and ask quotes.

This is why mid-price returns are often used for quote-based microstructure diagnostics.

However, mid-prices are not directly executable prices. A real trade pays spread and slippage.

In [ ]:
sample = quotes_l1.iloc[:2_000].copy()

plt.figure(figsize=(12, 6))
plt.plot(sample["timestamp"], sample["bid1"], label="bid1", alpha=0.7)
plt.plot(sample["timestamp"], sample["ask1"], label="ask1", alpha=0.7)
plt.plot(sample["timestamp"], sample["mid"], label="mid", linewidth=2)
plt.title("L1 Best Bid, Best Ask, and Mid-Price")
plt.xlabel("Timestamp")
plt.ylabel("Price")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(quotes_l1["spread"], bins=30)
plt.title("Quoted Spread Distribution")
plt.xlabel("Spread")
plt.ylabel("Quote count")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(quotes_l1["quote_interval_seconds"].dropna().clip(upper=60), bins=60)
plt.title("Quote Inter-Arrival Times, Clipped at 60 Seconds")
plt.xlabel("Seconds since previous quote")
plt.ylabel("Quote count")
plt.show()

## 9. Bar construction methods

We compare three bar types that can be constructed from L1 quote data.

### 9.1 Time bars

Sample the latest quote at fixed time intervals, such as 1 minute or 5 minutes.

### 9.2 Quote-count bars

Create a bar every $N$ quote updates.

This normalises by event count rather than clock time.

### 9.3 Range bars

Create a new bar whenever the mid-price moves by a specified number of ticks.

This normalises by price movement.

Important limitation:

> Without trade volume, we cannot construct true volume bars or dollar bars.

In [ ]:
def build_time_bars(
    quotes: pd.DataFrame,
    frequency: str,
    price_col: str = "mid"
) -> pd.DataFrame:
    """
    Build time bars using the last observed quote in each interval.
    """
    work = quotes.set_index("timestamp").sort_index().copy()

    agg = {
        price_col: ["first", "max", "min", "last"],
        "bid1": "last",
        "ask1": "last",
        "spread": "mean",
        "relative_spread": "mean",
        "symbol": "last",
        "session": "last"
    }

    bars = work.resample(frequency).agg(agg)
    bars.columns = ["_".join(col).strip("_") for col in bars.columns]
    bars = bars.dropna(subset=[f"{price_col}_last"]).reset_index()

    bars = bars.rename(columns={
        f"{price_col}_first": "open",
        f"{price_col}_max": "high",
        f"{price_col}_min": "low",
        f"{price_col}_last": "close",
        "bid1_last": "bid1_close",
        "ask1_last": "ask1_close",
        "spread_mean": "avg_spread",
        "relative_spread_mean": "avg_relative_spread",
        "symbol_last": "symbol",
        "session_last": "session"
    })

    bars["bar_type"] = f"time_{frequency}"
    bars["log_return"] = np.log(bars["close"]).diff()
    bars["num_quotes"] = work[price_col].resample(frequency).count().reindex(bars["timestamp"]).to_numpy()

    return bars


bars_1m = build_time_bars(quotes_l1, "1min")
bars_5m = build_time_bars(quotes_l1, "5min")
bars_15m = build_time_bars(quotes_l1, "15min")

bars_1m.head()

In [ ]:
def build_quote_count_bars(
    quotes: pd.DataFrame,
    quotes_per_bar: int,
    price_col: str = "mid"
) -> pd.DataFrame:
    """
    Build bars with a fixed number of quote updates per bar.
    """
    work = quotes.sort_values("timestamp").copy().reset_index(drop=True)
    work["bar_id"] = np.arange(len(work)) // quotes_per_bar

    grouped = work.groupby("bar_id")

    bars = grouped.agg(
        timestamp=("timestamp", "last"),
        open=(price_col, "first"),
        high=(price_col, "max"),
        low=(price_col, "min"),
        close=(price_col, "last"),
        bid1_close=("bid1", "last"),
        ask1_close=("ask1", "last"),
        avg_spread=("spread", "mean"),
        avg_relative_spread=("relative_spread", "mean"),
        num_quotes=(price_col, "count"),
        session=("session", "last"),
        symbol=("symbol", "last")
    ).reset_index(drop=True)

    bars["bar_type"] = f"quote_count_{quotes_per_bar}"
    bars["log_return"] = np.log(bars["close"]).diff()

    return bars


bars_q100 = build_quote_count_bars(quotes_l1, quotes_per_bar=100)
bars_q500 = build_quote_count_bars(quotes_l1, quotes_per_bar=500)

bars_q100.head()

In [ ]:
def build_range_bars(
    quotes: pd.DataFrame,
    range_ticks: int,
    tick_size: float,
    price_col: str = "mid"
) -> pd.DataFrame:
    """
    Build range bars: close a bar when mid-price range exceeds threshold.
    """
    threshold = range_ticks * tick_size

    rows = []
    current_quotes = []

    current_open = None
    current_high = None
    current_low = None

    for _, row in quotes.sort_values("timestamp").iterrows():
        price = float(row[price_col])

        if current_open is None:
            current_open = price
            current_high = price
            current_low = price
            current_quotes = [row]
            continue

        current_quotes.append(row)
        current_high = max(current_high, price)
        current_low = min(current_low, price)

        if current_high - current_low >= threshold:
            q = pd.DataFrame(current_quotes)

            rows.append({
                "timestamp": q["timestamp"].iloc[-1],
                "open": float(q[price_col].iloc[0]),
                "high": float(q[price_col].max()),
                "low": float(q[price_col].min()),
                "close": float(q[price_col].iloc[-1]),
                "bid1_close": float(q["bid1"].iloc[-1]),
                "ask1_close": float(q["ask1"].iloc[-1]),
                "avg_spread": float(q["spread"].mean()),
                "avg_relative_spread": float(q["relative_spread"].mean()),
                "num_quotes": int(len(q)),
                "session": q["session"].iloc[-1],
                "symbol": q["symbol"].iloc[-1],
            })

            current_open = None
            current_high = None
            current_low = None
            current_quotes = []

    bars = pd.DataFrame(rows)

    if bars.empty:
        return bars

    bars["bar_type"] = f"range_{range_ticks}_ticks"
    bars["log_return"] = np.log(bars["close"]).diff()

    return bars


bars_r5 = build_range_bars(quotes_l1, range_ticks=5, tick_size=config.tick_size)
bars_r10 = build_range_bars(quotes_l1, range_ticks=10, tick_size=config.tick_size)

bars_r5.head()

## 10. Previous-tick versus next-tick sampling

When sampling at fixed timestamps, there are two common choices:

1. **previous-tick sampling**: use the most recent quote available at or before the sample time;
2. **next-tick sampling**: use the next quote after the sample time.

Previous-tick sampling is usually safer for backtesting because it respects information availability.

Next-tick sampling can introduce look-ahead bias if used as if the quote were known at the sample time.

In [ ]:
def sample_previous_or_next_tick(
    quotes: pd.DataFrame,
    frequency: str,
    method: str = "previous"
) -> pd.DataFrame:
    """
    Sample quotes at fixed timestamps using previous-tick or next-tick interpolation.
    """
    q = quotes.sort_values("timestamp").copy()
    grid = pd.DataFrame({
        "sample_time": pd.date_range(
            start=q["timestamp"].min().floor(frequency),
            end=q["timestamp"].max().ceil(frequency),
            freq=frequency
        )
    })

    if method == "previous":
        sampled = pd.merge_asof(
            grid,
            q,
            left_on="sample_time",
            right_on="timestamp",
            direction="backward"
        )
    elif method == "next":
        sampled = pd.merge_asof(
            grid,
            q,
            left_on="sample_time",
            right_on="timestamp",
            direction="forward"
        )
    else:
        raise ValueError("method must be 'previous' or 'next'.")

    sampled = sampled.dropna(subset=["mid"]).copy()
    sampled["method"] = method
    sampled["staleness_seconds"] = (sampled["sample_time"] - sampled["timestamp"]).dt.total_seconds()
    sampled["lookahead_seconds"] = (sampled["timestamp"] - sampled["sample_time"]).dt.total_seconds()
    sampled["log_return"] = np.log(sampled["mid"]).diff()

    return sampled


previous_1m = sample_previous_or_next_tick(quotes_l1, "1min", method="previous")
next_1m = sample_previous_or_next_tick(quotes_l1, "1min", method="next")

previous_1m.head()

In [ ]:
sampling_comparison = pd.DataFrame({
    "method": ["previous_tick", "next_tick"],
    "num_samples": [len(previous_1m), len(next_1m)],
    "mean_abs_return": [
        previous_1m["log_return"].abs().mean(),
        next_1m["log_return"].abs().mean()
    ],
    "realized_variance": [
        np.sum(previous_1m["log_return"].dropna() ** 2),
        np.sum(next_1m["log_return"].dropna() ** 2)
    ],
    "mean_staleness_or_lookahead_seconds": [
        previous_1m["staleness_seconds"].abs().mean(),
        next_1m["lookahead_seconds"].abs().mean()
    ]
})

sampling_comparison

## 11. Bid/ask bounce demonstration

If a price series alternates between bid and ask, it can create artificial negative autocorrelation.

With L1 quote data, the mid-price is usually a cleaner quote-based price proxy:

$$
m_t = \frac{b_t+a_t}{2}
$$
This does not mean the mid is executable. It means the mid is often better for measuring quote-driven price movement.

In [ ]:
bounce_demo = quotes_l1.iloc[:5_000].copy()
bounce_demo["alternating_bid_ask_price"] = np.where(
    np.arange(len(bounce_demo)) % 2 == 0,
    bounce_demo["bid1"],
    bounce_demo["ask1"]
)

bounce_demo["alternating_log_return"] = np.log(bounce_demo["alternating_bid_ask_price"]).diff()
bounce_demo["mid_log_return"] = np.log(bounce_demo["mid"]).diff()

bounce_autocorr = pd.Series({
    "alternating_bid_ask_return_autocorr_lag1": bounce_demo["alternating_log_return"].dropna().autocorr(lag=1),
    "mid_return_autocorr_lag1": bounce_demo["mid_log_return"].dropna().autocorr(lag=1)
})

bounce_autocorr

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(
    bounce_demo["alternating_log_return"].dropna().clip(-0.005, 0.005),
    bins=80,
    alpha=0.6,
    label="Alternating bid/ask returns"
)
plt.hist(
    bounce_demo["mid_log_return"].dropna().clip(-0.005, 0.005),
    bins=80,
    alpha=0.6,
    label="Mid returns"
)
plt.title("Bid/Ask Bounce Demonstration")
plt.xlabel("Log return, clipped for display")
plt.ylabel("Count")
plt.legend()
plt.show()

## 12. Volatility signature plot

Microstructure noise tends to distort volatility estimates at very high sampling frequencies.

A volatility signature plot measures realised variance as a function of sampling interval:

$$
\begin{aligned}
RV(\Delta) &= \sum_i r_{i,\Delta}^2
\end{aligned}
$$
where $r_{i,\Delta}$ are returns sampled every $\Delta$.

If realised variance changes dramatically with sampling frequency, the sampling scheme is affecting the volatility estimate.

In [ ]:
def realized_variance_from_time_bars(quotes: pd.DataFrame, frequencies: list[str]) -> pd.DataFrame:
    """
    Compute realised variance across different time-bar frequencies.
    """
    rows = []

    for freq in frequencies:
        bars = build_time_bars(quotes, freq)
        returns = bars["log_return"].dropna()
        rv = float(np.sum(returns ** 2))
        ann_vol_proxy = np.sqrt(rv * 252 / max(config.num_business_days, 1))

        rows.append({
            "frequency": freq,
            "num_bars": len(bars),
            "realized_variance": rv,
            "annualized_vol_proxy": ann_vol_proxy,
            "mean_abs_return": float(returns.abs().mean())
        })

    return pd.DataFrame(rows)


signature_frequencies = ["5s", "15s", "30s", "1min", "2min", "5min", "10min", "15min"]
signature_df = realized_variance_from_time_bars(quotes_l1, signature_frequencies)

signature_df

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(signature_df["frequency"], signature_df["realized_variance"], marker="o")
plt.title("Volatility Signature Plot from L1 Mid-Price Time Bars")
plt.xlabel("Sampling frequency")
plt.ylabel("Realized variance")
plt.show()

## 13. Sampling bias comparison across bar types

We compare time bars, quote-count bars, and range bars using simple diagnostics:

1. number of bars;
2. mean absolute return;
3. realised variance;
4. return autocorrelation;
5. average spread;
6. average quotes per bar.

Different bars produce different statistical objects.

A strategy trained on one bar type may not behave the same on another.

In [ ]:
def bar_diagnostics(bars: pd.DataFrame, label: str) -> dict:
    """
    Compute diagnostic statistics for a bar DataFrame.
    """
    returns = bars["log_return"].replace([np.inf, -np.inf], np.nan).dropna()

    return {
        "bar_set": label,
        "num_bars": int(len(bars)),
        "mean_abs_return": float(returns.abs().mean()),
        "realized_variance": float(np.sum(returns ** 2)),
        "return_autocorr_lag1": float(returns.autocorr(lag=1)) if len(returns) > 2 else np.nan,
        "avg_spread": float(bars["avg_spread"].mean()) if "avg_spread" in bars else np.nan,
        "avg_quotes_per_bar": float(bars["num_quotes"].mean()) if "num_quotes" in bars else np.nan,
    }


bar_sets = {
    "time_1m": bars_1m,
    "time_5m": bars_5m,
    "time_15m": bars_15m,
    "quote_100": bars_q100,
    "quote_500": bars_q500,
    "range_5_ticks": bars_r5,
    "range_10_ticks": bars_r10,
}

bar_diagnostics_df = pd.DataFrame([
    bar_diagnostics(bars, label)
    for label, bars in bar_sets.items()
    if bars is not None and not bars.empty
])

bar_diagnostics_df

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = bar_diagnostics_df.sort_values("realized_variance", ascending=True)
plt.barh(plot_df["bar_set"], plot_df["realized_variance"])
plt.title("Realized Variance by Bar Construction")
plt.xlabel("Realized variance")
plt.ylabel("Bar set")
plt.show()

## 14. Session-break handling

Session gaps are dangerous.

If the market closes for lunch or overnight, a return from the last quote before the break to the first quote after the break mixes:

- market movement during the break;
- liquidity reopening effects;
- timestamp gap effects;
- possibly different trading sessions.

Depending on the research question, you may want to:

1. keep session-gap returns;
2. remove session-gap returns;
3. model them separately;
4. create separate session-level features.

This notebook flags bars where the time gap from previous bar is large.

In [ ]:
def flag_session_gaps(bars: pd.DataFrame, max_gap_minutes: float = 20.0) -> pd.DataFrame:
    """
    Flag bar returns that cross long time gaps.
    """
    out = bars.sort_values("timestamp").copy()
    out["bar_gap_minutes"] = out["timestamp"].diff().dt.total_seconds() / 60
    out["crosses_large_gap"] = out["bar_gap_minutes"] > max_gap_minutes
    out["log_return_no_large_gap"] = out["log_return"].where(~out["crosses_large_gap"])

    return out


bars_1m_gap = flag_session_gaps(bars_1m, max_gap_minutes=20)

bars_1m_gap[["timestamp", "log_return", "bar_gap_minutes", "crosses_large_gap"]].head()

In [ ]:
gap_effect = pd.Series({
    "num_1m_bars": len(bars_1m_gap),
    "large_gap_bars": int(bars_1m_gap["crosses_large_gap"].sum()),
    "rv_with_gaps": float(np.sum(bars_1m_gap["log_return"].dropna() ** 2)),
    "rv_without_large_gap_returns": float(np.sum(bars_1m_gap["log_return_no_large_gap"].dropna() ** 2)),
})

gap_effect

## 15. Optional top-of-book imbalance if sizes exist

If the dataset includes `bid1_size` and `ask1_size`, then a top-of-book imbalance can be computed:

$$
\begin{aligned}
I_t &= \frac{Q_t^{bid1} - Q_t^{ask1}} {Q_t^{bid1}+Q_t^{ask1}}
\end{aligned}
$$
This is **not** full order-book imbalance.

It uses only the displayed size at the best bid and best ask.

If sizes are unavailable, this section should be skipped.

In [ ]:
if {"bid1_size", "ask1_size", "top_of_book_imbalance"}.issubset(quotes_l1.columns):
    imbalance_summary = quotes_l1["top_of_book_imbalance"].describe()
    display(imbalance_summary)

    plt.figure(figsize=(10, 6))
    plt.hist(quotes_l1["top_of_book_imbalance"].dropna(), bins=60)
    plt.title("Optional Top-of-Book Imbalance Distribution")
    plt.xlabel("Top-of-book imbalance")
    plt.ylabel("Quote count")
    plt.show()
else:
    print("Bid/ask sizes unavailable; top-of-book imbalance section skipped.")

## 16. Short-horizon predictive diagnostic

With L1 quotes, a reasonable toy diagnostic is:

> Does current spread or top-of-book imbalance relate to the next mid-price move?

This is not a full alpha test. It is a microstructure sanity check.

For this synthetic dataset, any relationship is controlled by the simulation and should not be interpreted as evidence about real Chinese futures markets.

In [ ]:
def short_horizon_quote_diagnostic(quotes: pd.DataFrame, horizon_ticks: int = 20) -> pd.DataFrame:
    """
    Compute simple correlations between L1 features and future mid-price returns.
    """
    out = quotes.sort_values("timestamp").copy()
    out["future_log_mid"] = out["log_mid"].shift(-horizon_ticks)
    out["future_mid_return"] = out["future_log_mid"] - out["log_mid"]

    feature_cols = ["spread", "relative_spread"]

    if "top_of_book_imbalance" in out.columns:
        feature_cols.append("top_of_book_imbalance")

    rows = []

    for feature in feature_cols:
        valid = out[[feature, "future_mid_return"]].replace([np.inf, -np.inf], np.nan).dropna()
        corr = valid[feature].corr(valid["future_mid_return"])

        rows.append({
            "feature": feature,
            "horizon_ticks": horizon_ticks,
            "correlation_with_future_mid_return": corr,
            "n_obs": len(valid)
        })

    return pd.DataFrame(rows)


short_horizon_diagnostics = short_horizon_quote_diagnostic(quotes_l1, horizon_ticks=20)

short_horizon_diagnostics

## 17. Saving bar datasets and audit report

The notebook saves:

1. validated L1 quote data with derived fields;
2. time bars;
3. quote-count bars;
4. range bars;
5. sampling diagnostics;
6. volatility signature plot data;
7. an audit manifest.

These outputs can feed later alpha research notebooks.

In [ ]:
output_dir = Path("data/processed/tick_to_bar_sampling_bias_v1")
output_dir.mkdir(parents=True, exist_ok=True)

quotes_path = output_dir / "synthetic_l1_quotes_with_features.csv"
bars_1m_path = output_dir / "time_bars_1m.csv"
bars_5m_path = output_dir / "time_bars_5m.csv"
bars_q100_path = output_dir / "quote_count_bars_100.csv"
bars_r5_path = output_dir / "range_bars_5_ticks.csv"
diagnostics_path = output_dir / "bar_sampling_diagnostics.csv"
signature_path = output_dir / "volatility_signature.csv"
validation_path = output_dir / "l1_validation_report.csv"
manifest_path = output_dir / "manifest.json"

quotes_l1.to_csv(quotes_path, index=False)
bars_1m.to_csv(bars_1m_path, index=False)
bars_5m.to_csv(bars_5m_path, index=False)
bars_q100.to_csv(bars_q100_path, index=False)
bars_r5.to_csv(bars_r5_path, index=False)
bar_diagnostics_df.to_csv(diagnostics_path, index=False)
signature_df.to_csv(signature_path, index=False)
validation_report.to_csv(validation_path, index=False)

manifest = {
    "dataset_name": "synthetic_l1_tick_to_bar_sampling_bias",
    "schema_version": "tick_to_bar_sampling_bias_v1",
    "created_at": pd.Timestamp.now().isoformat(),
    "config": asdict(config),
    "data_scope": "L1 top-of-book quotes: bid1 and ask1. Optional bid1_size and ask1_size included only for demonstration.",
    "not_supported_without_l2": [
        "full order book depth",
        "queue position",
        "multi-level order imbalance",
        "full cancellation dynamics",
        "true depth-aware microprice"
    ],
    "saved_files": {
        "quotes": str(quotes_path),
        "time_bars_1m": str(bars_1m_path),
        "time_bars_5m": str(bars_5m_path),
        "quote_count_bars_100": str(bars_q100_path),
        "range_bars_5_ticks": str(bars_r5_path),
        "diagnostics": str(diagnostics_path),
        "volatility_signature": str(signature_path),
        "validation": str(validation_path)
    }
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

quotes_path, diagnostics_path, manifest_path

## 18. Practical checklist for L1 tick-to-bar research

Before using bars built from `bid1` and `ask1`, check:

1. **Quote validity**  
   Are there crossed, locked, missing, or non-positive quotes?

2. **Price proxy**  
   Are returns based on bid, ask, mid, or last trade? With only L1 quotes, mid is often the cleanest quote proxy.

3. **Executability**  
   Mid-price is not executable. Spread and slippage must be modelled separately.

4. **Sampling rule**  
   Are bars time-based, quote-count-based, range-based, or session-based?

5. **Interpolation method**  
   Does sampling use previous tick or next tick? Next tick can leak future information.

6. **Session breaks**  
   Are lunch, overnight, and night-session gaps handled deliberately?

7. **Spread regime**  
   Are wide-spread periods included, filtered, or modelled separately?

8. **Sampling frequency**  
   Does realised volatility change materially across frequencies?

9. **Data limitation**  
   Are claims limited to L1 quote data? Avoid saying “order-book imbalance” if only bid1/ask1 prices exist.

10. **Downstream consistency**  
   Does the backtester use the same price convention as the feature pipeline?

## 19. Limitations

This notebook is designed around L1 quote data.

### 19.1 No full limit order book

With only `bid1` and `ask1`, the notebook cannot model:

- depth beyond the best level;
- queue position;
- hidden liquidity;
- order cancellations across levels;
- price impact through the book;
- true multi-level imbalance.

Any later notebook using this data should say **top-of-book**, not full LOB.

### 19.2 No trade data by default

If the dataset lacks trade prints, we cannot compute:

- trade direction;
- buyer-initiated versus seller-initiated flow;
- realised spread;
- effective spread;
- VWAP;
- true volume bars;
- dollar bars.

If last trade price and volume are available, those can be added as optional fields.

### 19.3 Synthetic sessions are simplified

The session windows in this notebook are illustrative.

Real Chinese futures products have product-specific day and night sessions, holidays, breaks, and exchange rules.

### 19.4 Mid-price is not executable

Mid-price returns are useful for measuring quote movement, but a strategy cannot generally trade at the mid.

Execution models must include spread crossing, queueing, latency, and slippage assumptions.

### 19.5 Sampling can change the answer

Different bar construction methods produce different return distributions, volatility estimates, and autocorrelations.

A model trained on one bar type may not generalise to another.

### 19.6 Microstructure noise is not eliminated

Using mid-price and coarser bars can reduce some noise, but it does not eliminate microstructure effects.

Volatility signature plots and robustness checks are still necessary.

## 20. Research frontier and current directions

L1 tick-to-bar research sits between data engineering and market microstructure.

### 20.1 Quote-based realised volatility

High-frequency realised volatility is sensitive to microstructure noise.

Volatility signature plots help identify sampling frequencies where realised variance stabilises.

A future notebook could compare standard realised variance with noise-robust estimators such as realised kernels or pre-averaging.

### 20.2 Event-based bars

Time bars ignore variation in market activity.

Event-based bars such as tick bars, quote-count bars, imbalance bars, and range bars attempt to sample when information arrives rather than when the clock ticks.

With only L1 quotes, quote-count and range bars are feasible. True volume/dollar bars require trade volume.

### 20.3 Top-of-book dynamics

Even without full LOB depth, best bid and ask dynamics can be modelled.

A future notebook could model spread changes, mid-price jumps, and quote arrivals using point processes such as Hawkes processes.

### 20.4 Exchange-specific futures microstructure

Chinese futures markets have features that matter for L1 data:

- night sessions;
- lunch breaks;
- price limits;
- product-specific tick sizes;
- contract-specific liquidity migration;
- dominant-contract conventions;
- exchange calendar effects.

A future notebook could create an exchange calendar adapter for DCE, CZCE, SHFE, and INE products.

### 20.5 From L1 to execution-aware backtesting

Even L1 data can improve execution assumptions compared with daily bars.

A future backtester could model:

- buying at ask1;
- selling at bid1;
- spread cost;
- stale quote filtering;
- bar-close execution delay;
- no-trade rules during wide spreads.

This would be more realistic than assuming all trades occur at mid or close.

### 20.6 Optional extension if size fields exist

If `bid1_size` and `ask1_size` are available, the repo can add:

- top-of-book imbalance;
- simple microprice;
- queue-pressure proxies;
- imbalance-conditioned short-horizon returns.

But this should be described as **top-of-book imbalance**, not full order-book imbalance.

## 21. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_08_l1_quote_imbalance_and_microprice_optional**  
   Using bid/ask prices and optional sizes for top-of-book features.

2. **03_14_information_coefficient_and_feature_decay**  
   Testing whether L1 spread or quote-count features predict future mid returns.

3. **05_02_event_driven_backtest_engine**  
   Building an event-driven backtester that consumes L1 quote updates.

4. **05_03_transaction_costs_slippage_latency**  
   Modelling bid/ask spread crossing and bar-close execution delay.

5. **06_04_hawkes_process_order_flow**  
   Modelling best bid/ask update events as point processes.

6. **06_06_microstructure_friction_cpp_core**  
   Moving high-frequency quote processing into a compiled kernel.

7. **07_01_multi_asset_cta_research_pipeline**  
   Combining L1-derived bar data with futures rolling, risk sizing, and backtesting.

## 22. Summary

This notebook restructured the tick-to-bar module around the actual data constraint:

> L1 top-of-book quotes: `bid1` and `ask1`.

It showed how to:

1. validate L1 quote data;
2. compute mid-price and spread;
3. analyse quote update intensity;
4. build time bars;
5. build quote-count bars;
6. build range bars;
7. compare previous-tick and next-tick sampling;
8. demonstrate bid/ask bounce;
9. produce volatility signature plots;
10. flag session gaps;
11. optionally use top-of-book size imbalance if bid/ask sizes exist;
12. save bar datasets and diagnostics.

The key computational takeaway is:

> Tick-to-bar conversion is a sampling operator. It changes the statistical properties of the data.

The key financial takeaway is:

> L1 quote data is valuable, but it should be described honestly. It supports top-of-book microstructure analysis, not full limit-order-book modelling.

## 23. Further reading

### Market microstructure and L1 quotes

- Literature on bid-ask bounce and market microstructure noise.
- Hansen and Lunde on realised variance and market microstructure noise.
- Volatility signature plots in high-frequency finance.
- Top-of-book versus depth-of-book market data documentation.

### Bar construction

- Time bars, tick bars, volume bars, dollar bars, and range bars.
- Event-based sampling in financial machine learning.
- Imbalance and run bars, when trade signs and volumes are available.

### Chinese futures extensions

- Product-specific trading calendars for DCE, CZCE, SHFE, and INE.
- Chinese futures tick-size and price-limit metadata.
- Night-session handling.
- Dominant-contract construction.
- L1 quote-based execution modelling.